In [1]:
!pip install --upgrade langchain
!pip install python-dotenv
!pip install openai

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached xxhash-3.6.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 13.6 MB/s eta 0:00:00
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using c

In [6]:
!pip install --upgrade openai python-dotenv

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [7]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [ ]:
This is a function that gets a  pizza name and returns its price 

In [9]:
def get_pizza_info(pizza_name: str):
    menu = {
        "salami": 10.99,
        "margherita": 9.49,
        "pepperoni": 11.99,
        "hawaii": 11.49,
    }

    key = pizza_name.strip().lower()
    price = menu.get(key)

    if price is None:
        return json.dumps({
            "name": pizza_name,
            "available": False,
            "price": None,
            "message": f"{pizza_name} is not on the menu."
        })

    return json.dumps({
        "name": pizza_name,
        "available": True,
        "price": price
    })

In [ ]:
describing my tool to the model 

In [12]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_pizza_info",
            "description": "Get the name, availability, and price of a pizza from the restaurant menu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pizza_name": {
                        "type": "string",
                        "description": "The exact pizza name, for example Salami or Margherita."
                    }
                },
                "required": ["pizza_name"]
            }
        }
    }
]

In [ ]:
A question that should nt use the tool is here : 

In [14]:
MODEL="gpt-4o-mini"

In [15]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of Morocco?"}],
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message
print("content:", message.content)
print("tool_calls:", message.tool_calls)

content: The capital of Morocco is Rabat.
tool_calls: None


In [18]:
query ="How much does pizza salami cost?"

In [20]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": query}],
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message
print("content:", message.content)
print("tool_calls:", message.tool_calls)

content: None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_8KoNhXrMRilCbVxcSHziiV2y', function=Function(arguments='{"pizza_name":"Salami"}', name='get_pizza_info'), type='function')]


In [21]:
print(message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_8KoNhXrMRilCbVxcSHziiV2y', function=Function(arguments='{"pizza_name":"Salami"}', name='get_pizza_info'), type='function')]


In [23]:
import json
tool_call = message.tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print("function_name:", function_name)
print("arguments:", arguments)

if function_name == "get_pizza_info":
    function_response = get_pizza_info(arguments["pizza_name"])
else:
    raise ValueError(f"Unknown function: {function_name}")

print(function_response)

function_name: get_pizza_info
arguments: {'pizza_name': 'Salami'}
{"name": "Salami", "available": true, "price": 10.99}


In [ ]:
# I am sending the result back so I can get the real answer 

In [24]:
second_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": query},
        message,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": function_response,
        },
    ],
    tools=tools,
)

final_message = second_response.choices[0].message
print(final_message.content)

The pizza Salami costs $10.99 and is currently available.


In [26]:
# wrap it in one reusable function 

def ask_with_tools(query: str):
    messages = [{"role": "user", "content": query}]

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            return message.content, messages

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if name == "get_pizza_info":
                result = get_pizza_info(args["pizza_name"])
            else:
                result = json.dumps({"error": f"Unknown tool: {name}"})

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })
    
    

In [38]:
print(ask_with_tools("I want two pizzas : one Salami and one marguerita how much will I pay?"))
print(ask_with_tools("What is the capital of France?"))

('The prices for the pizzas are as follows:\n\n- Salami Pizza: $10.99\n- Margherita Pizza: $9.49\n\nThe total cost will be **$20.48**.', [{'role': 'user', 'content': 'I want two pizzas :one Salami and one marguerita how much will I pay?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_FOjsaad55erWEQvvH4WqNu5b', function=Function(arguments='{"pizza_name": "Salami"}', name='get_pizza_info'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_3EboCwsk3wAA77XN0muNf4sy', function=Function(arguments='{"pizza_name": "Margherita"}', name='get_pizza_info'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_FOjsaad55erWEQvvH4WqNu5b', 'content': '{"name": "Salami", "available": true, "price": 10.99}'}, {'role': 'tool', 'tool_call_id': 'call_3EboCwsk3wAA77XN0muNf4sy', 'content': '{"name": "Margherita", "available": true, "price": 9.49}'}, ChatCompl

In [39]:
from langchain.tools import BaseTool

In [51]:
from typing import Optional

from langchain.tools import BaseTool
from langchain_core.callbacks import CallbackManagerForToolRun


class StupidJokeTool(BaseTool):
    name: str = "StupidJokeTool"
    description: str = "Tool to explain jokes about chickens"

    def _run(
        self,
        query: str,
        run_manager: Optional[CallbackManagerForToolRun] = None,
    ) -> str:
        return "It is funny, because AI..."

In [52]:
from langchain.tools import format_tool_to_openai_function, MoveFileTool

tools = [StupidJokeTool(), MoveFileTool()]
functions = [format_tool_to_openai_function(t) for t in tools]

ImportError: cannot import name 'format_tool_to_openai_function' from 'langchain.tools' (/home/u991938/Sarah/personalization_mvp/.venv/lib/python3.12/site-packages/langchain/tools/__init__.py)

In [55]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

ModuleNotFoundError: No module named 'langchain_openai'

In [62]:
%pip install -U langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 797.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.5/801.5 kB 26.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [68]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

In [69]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [71]:
from langchain.tools import tool

@tool
def explain_chicken_joke(query: str) -> str:
    """Explain jokes about chickens."""
    return "It is funny because the joke uses a simple setup and a familiar punchline."

In [72]:
llm_with_tools = llm.bind_tools([explain_chicken_joke])
ai_msg = llm_with_tools.invoke("Why does the chicken cross the road?")
print(ai_msg.tool_calls)

[{'name': 'explain_chicken_joke', 'args': {'query': 'Why does the chicken cross the road?'}, 'id': 'call_ZMGj2bQVQN9iut3diYU6KeT1', 'type': 'tool_call'}]


In [77]:
messages = [HumanMessage(content=query), ai_msg]

for tool_call in ai_msg.tool_calls:
    
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print("function_name:", function_name)
    print("arguments:", arguments)
    

AttributeError: 'dict' object has no attribute 'function'

In [79]:
import json

tool_call = message.tool_calls[0]



In [83]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ChatMessage, ToolMessage

In [85]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
message = llm.invoke([HumanMessage(content="What is the capital of France?")])

print(message.content)

The capital of France is Paris.


In [ ]:
functions = [
    {
        "name": "get_pizza_info",
        "description": "Get name and price of a pizza of the restaurant",
        "parameters": {
            "type": "object",
            "properties": {
                "pizza_name": {
                    "type": "string",
                    "description": "The name of the pizza, e.g. Salami",
                },
            },
            "required": ["pizza_name"],
        },
    }
]